# RSNA Competition

### 1. ACL
What the CNN looks for on MRI:
* A healthy ACL appears as a tight, dark (hypointense) band of straight fibers on Sagittal MRI slices.
* An ACL tear shows up as a disrupted, wavy, swollen, or hyperintense (bright/white signal) blob where the dark band should be.
### 2. MCL
What the CNN looks for on MRI:
* Best evaluated on Coronal MRI slices (front view).
* A torn MCL shows fluid leakage (bright white signal on fluid-sensitive sequences) surrounding or completely severing the dark medial ligament line.
### 3. OA
What the CNN looks for on MRI:
* **Joint Space Narrowing:** The gap between bones shrinks as cartilage degrades.
* **Cartilage Loss:** The thin grey cartilage layer lining the bone edges becomes thin, ragged, or completely missing.
* **Osteophytes:** Bony outgrowths (spurs) sticking off the edges of the joint.
* **Subchondral Sclerosis / Bone Edema:** Bright white spots inside the bone tissue right underneath the worn cartilage, showing bone distress from friction.
### 4. Effusion:
What the CNN looks for on MRI:
* **Best View**: Sagittal T2 / PD Fat-Suppressed.
* **Normal:** The joint capsule should appear tight, and the space between bones should be uniformly dark.
* **Abnormal (Effusion):** Excess fluid appears as bright white, blob-like signal accumulation within the joint capsule, often obscuring the normal joint anatomy.
### 5. Synovitis:
* Effusion vs. Synovitis:

    * **Effusion** = The fluid filling the room.

    * **Synovitis** = The walls/lining of the room becoming thick, swollen, and inflamed.

* **What the CNN looks for on MRI:**
    * Instead of clean liquid, you'll see thick, shaggy, frond-like tissue lining the joint space or floating inside the fluid pouch.
    * On contrast-enhanced T1 images or T2 sequences, this thickened lining lights up intensely along the joint boundaries.

### 6. Baker's Cyst:
What the CNN looks for on MRI:
* **Best Views**: Axial (top-down slice) and Sagittal.
* **Abnormal (Baker's Cyst):** A **well-defined, comma-shaped or tear-shaped bright white fluid collection** nestled between the posterior muscle bellies.

### 7. Contusion:
What the CNN looks for on MRI:
* **T2 / Fat-Suppressed (PD-FS) Sequences:** Healthy bone marrow appears dark/grey. A bone contusion shows up as an **ill-defined, bright white cloud** (hyperintensity) glowing inside the marrow cavity.

---

## Part 2: Reading DICOM Planes for Your CNN Pipeline

An MRI scan isn't a single 2D image; it's a 3D volume sampled along three principal orthogonal planes. When designing a multi-view model (e.g., 2.5D CNN or 3D ResNet/EfficientNet), mapping targets to the optimal view plane gives your feature extractor the highest signal-to-noise ratio.

```
       [ AXIAL ]               [ CORONAL ]              [ SAGITTAL ]
   Top-down cross section       Frontal view             Side profile
   (Slicing Thigh -> Calf)   (Slicing Front -> Back)   (Slicing Left -> Right)

```

### 1. Sagittal Plane (Side Profile)

* **Best for:** `ACL`, `Effusion`, `PF OA`, `Medial Meniscus` (anterior/posterior horns), `Lateral Meniscus`.
* **Why:** Lets the model trace the diagonal path of the ACL and evaluate the profile of the kneecap and suprapatellar fluid pouch.

### 2. Coronal Plane (Frontal View)

* **Best for:** `MCL`, `Medial Meniscus` (body), `Lateral Meniscus` (body), `Medial OA`, `Lateral OA`.
* **Why:** Provides a side-by-side view of both joint compartments (inner vs. outer) to measure joint space narrowing and side-wall ligament tears.

### 3. Axial Plane (Top-Down View)

* **Best for:** `PF OA`, `Baker's Cyst`, `Patella alignment`, `Contusion` localization.
* **Why:** Shows the posterior popliteal space behind the knee clearly (for Baker's Cyst) and the groove behind the kneecap.

---

## Complete 12-Target Cheat Sheet for Model Training

| Target | Category | Primary DICOM Plane | Key Target Signal |
| --- | --- | --- | --- |
| **`ACL`** | Ligament | Sagittal | Disrupted/bright signal in center ligament band |
| **`MCL`** | Ligament | Coronal | Thickening/bright fluid lining inner wall |
| **`Medial Meniscus`** | Cartilage | Sagittal / Coronal | Bright line intersecting inner triangle edge |
| **`Lateral Meniscus`** | Cartilage | Sagittal / Coronal | Bright line intersecting outer triangle edge |
| **`Medial OA`** | Degeneration | Coronal | Inner joint gap narrowing + cartilage loss |
| **`Lateral OA`** | Degeneration | Coronal | Outer joint gap narrowing + cartilage loss |
| **`PF OA`** | Degeneration | Axial / Sagittal | Narrowing behind patella + bone spurs |
| **`Effusion`** | Fluid | Sagittal | Bright white pool expanding suprapatellar pouch |
| **`Synovitis`** | Fluid / Inflam. | Sagittal / Axial | Thickened, shaggy lining inside fluid spaces |
| **`Baker's Cyst`** | Fluid | Axial / Sagittal | Bright fluid sac in posterior popliteal space |
| **`Contusion`** | Bone Trauma | Coronal / Sagittal | Glowing bright cloud inside dark bone marrow |
| **`Fracture`** | Bone Trauma | Coronal / Sagittal | Jagged dark cortical line + surrounding edema |

---


Listed directory RMSA
Viewed notes.ipynb:1-134
Searched web: ""RSNA Knee Abnormality Detection" kaggle"
Searched web: ""rsna-knee-abnormality-detection" OR "RSNA Knee Abnormality Detection" series slices kaggle"
Searched web: ""RSNA Knee Abnormality Detection" preprocessed dataset png npz kaggle"

Handling a massive **~570 GB DICOM MRI dataset** with limited resources (**Kaggle Notebooks** and a **6GB RTX 4050 laptop**) is one of the most classic challenges in medical computer vision competitions.

The good news is: **Top Kaggle Grandmasters almost never train models directly on raw 570GB DICOMs.** Instead, they use a structured methodology to shrink the data volume, divide the training into modular stages, and train lightweight, high-performance architectures.

Here is a breakdown of all the ways you can partition, compress, and train on this dataset within your hardware constraints.

---

```
                                  [ RAW DICOM: ~570 GB ]
                                             │
               ┌─────────────────────────────┴─────────────────────────────┐
               ▼                                                           ▼
    [ Route A: Community NPZ ]                                  [ Route B: Custom Chunker ]
   Attach public preprocessed 3D                               Parallel Kaggle background jobs
   volumes dataset (~15-25 GB)                                (e.g., 4 notebooks × 25% data)
               │                                                           │
               └─────────────────────────────┬─────────────────────────────┘
                                             ▼
                               [ PREPROCESSED DATA: ~20 GB ]
                                             │
               ┌─────────────────────────────┼─────────────────────────────┐
               ▼                             ▼                             ▼
       [ Plane Splitting ]           [ 5-Fold Splitting ]        [ 2-Stage Stacking ]
     • Model 1: Sagittal (ACL/Eff)   • Fold 0 Notebook (80/20)   • Stage 1: 2.5D Backbones
     • Model 2: Coronal (MCL/OA)     • Fold 1 Notebook (80/20)   • Stage 2: LightGBM / MLP
     • Model 3: Axial (Cyst/PFOA)    • Parallel Execution          on pooled features
```

---

### Strategy 1: Data Volume Reduction (Shrink 570 GB $\rightarrow$ 15–25 GB)

Raw DICOMs are massive because they contain uncompressed 16-bit pixel data, raw resolutions ($512 \times 512$), slice depth variations ($20$ to $100+$ slices per series), and header metadata.

#### 1. Resampling & Resolution Downscaling
* **Spatial Resize:** Downscale slices from $512 \times 512$ to **$224 \times 224$** or **$256 \times 256$**. In MRI, knee joint pathology (ligament tears, cartilage wear) remains clearly visible at $256 \times 256$.
* **Slice Depth Resampling:** Uniformly sample a fixed depth (e.g., **$24$ or $32$ slices**) per series.
* **8-bit / 16-bit Normalization:** Clip pixel intensity (1st to 99th percentile) and convert to `uint8` ($0-255$).
* **File Format:** Save each series as a compressed **`.npz`** or **`.h5`** file.

$$\text{Single Series Size} = 32 \text{ slices} \times 256 \times 256 \times 1 \text{ byte (uint8)} \approx \mathbf{2.09 \text{ MB}}$$
For $\sim 24,000$ series, the total dataset shrinks to **$\sim 18\text{--}25\text{ GB}$**, which easily fits into Kaggle's 73 GB disk space.

#### 2. How to obtain this without running out of Kaggle Disk
* **Option A (Instant - Recommended):** Kaggle has a public preprocessed dataset created for this competition: **`RSNA Knee MRI - Processed 3D Volumes`** (search in Kaggle Datasets). You can simply click **"Add Input"** to your notebook — it mounts virtually without taking up your notebook's disk space.
* **Option B (Custom Partitioning):** If you want your own custom preprocessing:
  1. Create 4 separate Kaggle notebooks.
  2. Notebook 1 processes `study_id` $0 \rightarrow 25\%$, Notebook 2 processes $25 \rightarrow 50\%$, etc.
  3. Save the outputs as 4 Kaggle Datasets (`rsna-knee-part-1`, `part-2`, etc.) and attach all 4 to your training notebook.

---

### Strategy 2: Splitting Methods for Training

Instead of training one monolithic model on the entire dataset at once, you can divide the problem along three natural axes:

#### 1. Split by Plane (Sagittal vs. Coronal vs. Axial)
As noted in your competition notes, knee pathologies are anatomically best visible from specific views:
* **Sagittal Model:** Trained only on sagittal series $\rightarrow$ outputs predictions for `ACL`, `Effusion`, `Medial Meniscus`, `Lateral Meniscus`.
* **Coronal Model:** Trained only on coronal series $\rightarrow$ outputs predictions for `MCL`, `Medial OA`, `Lateral OA`, `Contusion`.
* **Axial Model:** Trained only on axial series $\rightarrow$ outputs predictions for `PF OA`, `Baker's Cyst`, `Synovitis`.

**Benefits:**
* Each notebook only loads **$\sim 33\%$ of the series**.
* Training completes $3\times$ faster per model.
* Specialized feature extractors prevent gradient interference between conflicting anatomical views.

#### 2. Stratified Group K-Fold (Fold-by-Fold Training)
* Split the dataset into 5 folds grouped by `study_id` (to prevent data leakage across series of the same patient).
* Train **Fold 0** in **Notebook A**, **Fold 1** in **Notebook B**, etc.
* You can run multiple Kaggle background runs simultaneously.
* Save the weights (`fold0_best.pth`, `fold1_best.pth`, etc.) as dataset artifacts, then ensemble them during inference.

#### 3. Two-Stage Pipeline (Feature Extraction $\rightarrow$ Tabular Stacking)
* **Stage 1 (Feature Extractor):** Train 2.5D slice encoders on a subset or per-plane basis. Freeze the CNN and extract a 1D embedding vector (e.g., 512 dimensions) for each series/study.
* **Stage 2 (Meta-Classifier):** Train a **LightGBM / CatBoost / MLP** on the extracted embeddings + clinical text metadata. This stage trains in under 2 minutes on CPU or your RTX 4050 laptop.

---

### Strategy 3: Memory-Efficient Architectures for 6GB VRAM & Kaggle GPUs

To train 3D/2.5D MRI models without running out of CUDA memory (OOM), use the **2.5D CNN + Slice Aggregator** paradigm:

```
[ Input: (Batch, 24 Slices, 1, 256, 256) ]
                    │
                    ▼
[ Reshape to (Batch * 24, 1, 256, 256) ]
                    │
                    ▼
[ 2D Backbone (e.g. Timm EfficientNet-B0 / ResNet34) ]
                    │
                    ▼
[ Slices Features: (Batch, 24, 512) ]
                    │
                    ▼
[ Slice Aggregator: Attention Pooling / 1D Conv / Transformer / GRU ]
                    │
                    ▼
[ Study-Level Output: (Batch, 12 Targets) ]
```

#### PyTorch Memory Optimization Boilerplate
This configuration ensures training fits comfortably within your **RTX 4050 6GB VRAM** or Kaggle's T4 GPU:

```python
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

# 1. 2.5D Model Definition
class RSNAKnee25D(nn.Module):
    def __init__(self, backbone, num_classes=12):
        super().__init__()
        self.encoder = backbone  # e.g. timm.create_model('resnet34', num_classes=0)
        self.attn_pool = nn.Sequential(
            nn.Linear(512, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
            nn.Softmax(dim=1)
        )
        self.classifier = nn.Linear(512, num_classes)
        
    def forward(self, x):
        # x shape: (B, S, C, H, W) where S = num_slices (e.g. 24)
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        feats = self.encoder(x)  # (B * S, 512)
        feats = feats.view(B, S, -1)  # (B, S, 512)
        
        # Attention pooling across slices
        weights = self.attn_pool(feats)  # (B, S, 1)
        pooled = (feats * weights).sum(dim=1)  # (B, 512)
        return self.classifier(pooled)

# 2. Memory-Efficient Training Loop
scaler = GradScaler()
accum_steps = 4  # Simulates batch_size = 4 * 4 = 16

for step, (images, targets) in enumerate(dataloader):
    images = images.to(device)   # Small physical batch: 4
    targets = targets.to(device)
    
    with autocast(dtype=torch.float16): # Halves VRAM usage
        outputs = model(images)
        loss = criterion(outputs, targets) / accum_steps
        
    scaler.scale(loss).backward()
    
    if (step + 1) % accum_steps == 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
```

---

### Strategy 4: Division of Labor (Laptop vs. Kaggle Cloud)

| Task | Device | Why |
| :--- | :--- | :--- |
| **Pipeline & EDA Prototyping** | **RTX 4050 Laptop** | Fast iteration on 100 sample `.npz` files. Test loss functions, dataloaders, augmentations without waiting for Kaggle queues. |
| **Dataset Conversion** | **Kaggle (or Community NPZ)** | Download/process directly on Kaggle cloud or attach the pre-converted community dataset. |
| **Full Model Training** | **Kaggle Notebooks (T4 / P100)** | Run 5 folds or 3 plane models in parallel via background commits (`Save Version` $\rightarrow$ `Run all`). |
| **Meta-model / Stacking** | **RTX 4050 Laptop / Kaggle** | LightGBM/XGBoost on extracted embeddings runs in minutes locally. |
| **Final Inference Submission** | **Kaggle Submission Notebook** | Load saved weights and run test inference within Kaggle's 9-hour submission budget. |

---

### Suggested Next Steps

1. **Test Data Loading:** Would you like to inspect a starter script to load and visualize the `.npz` 3D series volumes (Sagittal, Coronal, Axial) using PyTorch?
2. **Build the Baseline:** We can create a modular training pipeline with cross-validation and attention pooling ready to run on Kaggle.